# Validate Pocket-3D Features

Sanity + mini-XGBoost gate **before** committing to the full clustered benchmark.

**What this notebook checks:**
1. Extraction ran cleanly (no NaN / inf, `has_pocket` coverage ≥850, `atom_completeness` mean ≥0.95).
2. Per-feature distributions are non-degenerate; high-correlation pairs flagged (|r|>0.95).
3. **Literature spot check** on 5 canonical FPs — EGFP OH↔H148-ND1 ≈ 3.0-3.5 Å, Sirius OH environment hydrophobic, DsRed phenol↔M163 <5 Å, etc.
4. **Correlation sanity**: Pearson r of `τ` dihedral vs QY, and `electrostatic_proxy_OH` vs ex_max. Both ~0 ⇒ features are noise.
5. **Mini-XGBoost gate** (the critical decision): 5-fold random KFold on ex_max with three subsets — `L_pocket3d_only`, `B_lora`, `B_lora+pocket3d`. If `B_lora+pocket3d` is ≥0.3 nm **worse** than `B_lora`, abort full clustered benchmark and enter fallback ladder.

**Runtime.** ~20 min on H100. Heavy because of the 5-fold Optuna-free XGBoost mini-run (cell 10).

In [ ]:
# ── Cell 1: Install + mount ────────────────────────────────────────────────────────────────────────────────────────
!pip install -q -U xgboost scikit-learn
import os, sys
from pathlib import Path

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR       = Path('/content/drive/Othercomputers/My Mac/FluorCode/data')
    STRUCT_DIR_PDB = DATA_DIR / 'minimized'
    STRUCT_DIR     = Path('/content/drive/MyDrive/FluorCode/struct_features')
    LORA_DIR       = Path('/content/drive/MyDrive/FluorCode/results_lora_esm2')
    REPO           = Path('/content/drive/MyDrive/FluorCode_colab')
else:
    # Local: assumes running from repository root
    REPO           = Path.cwd().parent.parent  # model/LoRA_ESM2_Structure -> repo root
    DATA_DIR       = REPO / 'data' / 'sequence'
    STRUCT_DIR_PDB = REPO / 'data' / 'structure' / 'minimized'
    STRUCT_DIR     = REPO / 'model' / 'LoRA_ESM2_Structure'
    LORA_DIR       = REPO / 'model' / 'LoRA_ESM2'

SEQ_DIR    = DATA_DIR
MODULE_DIR = REPO / 'model' / 'LoRA_ESM2_Structure'
sys.path.insert(0, str(MODULE_DIR))

assert (STRUCT_DIR / 'pocket3d_features.npz').exists(), \
    'Run build_pocket3d_features.ipynb first.'
assert (LORA_DIR / 'lora_embeddings_all_folds.npz').exists(), 'LoRA npz missing'
print('All inputs present.')

In [ ]:
# ── Cell 2: Load pocket3d + LoRA + meta ────────────────────────────────────────────────────────────────────────────
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')

d = np.load(STRUCT_DIR / 'pocket3d_features.npz', allow_pickle=True)
X_p3d_full = d['features'].astype(np.float32)
p3d_names  = list(d['feature_names'])
p3d_slugs  = list(d['slugs'])
has_pocket = d['has_pocket'].astype(np.int8)
atom_compl = d['atom_completeness'].astype(np.float32)
print(f'pocket3d : {X_p3d_full.shape}  has_pocket={int(has_pocket.sum())}/{len(has_pocket)}')

meta_full = pd.read_csv(SEQ_DIR / 'fp_embeddings_meta.csv')
meta = meta_full[meta_full['cofactor'].isna()].reset_index(drop=True)
meta['brightness'] = meta['qy'] * meta['ext_coeff'] / 1000.0
slugs = meta['slug'].tolist()
print(f'meta GFP-family: {len(slugs)}')

# LoRA fold-0 is already pre-filtered to the 927 GFP-family rows
# (matches the GFP-filtered meta row order). No re-indexing needed.
lora = np.load(LORA_DIR / 'lora_embeddings_all_folds.npz')
X_lora = lora['embeddings'][0].astype(np.float32)
print(f'LoRA fold-0    : {X_lora.shape}')
assert X_lora.shape[0] == len(slugs), (
    f'LoRA rows ({X_lora.shape[0]}) ≠ GFP meta rows ({len(slugs)}). '
    'LoRA was likely fine-tuned on a different cohort — verify lora_embeddings_all_folds.npz.'
)

# Align pocket3d to GFP-family slug order via slug lookup.
slug2p3d = {s: X_p3d_full[i]      for i, s in enumerate(p3d_slugs)}
slug2hp  = {s: int(has_pocket[i]) for i, s in enumerate(p3d_slugs)}
D_p = X_p3d_full.shape[1]
X_p3d = np.stack([slug2p3d.get(s, np.zeros(D_p, dtype=np.float32)) for s in slugs]).astype(np.float32)
has_p = np.array([slug2hp.get(s, 0) for s in slugs], dtype=np.float32)[:, None]
X_p3d[~np.isfinite(X_p3d)] = 0.0

print(f'aligned → X_p3d={X_p3d.shape}  X_lora={X_lora.shape}  has_p={has_p.shape}  '
      f'(has_pocket in GFP cohort: {int(has_p.sum())}/{len(slugs)})')

In [ ]:
# ── Cell 3: Basic integrity — NaN / inf / duplicate detection ──────────────────────────────────────────────────────────────
print('any NaN in pocket3d:', bool(np.isnan(X_p3d_full).any()))
print('any inf in pocket3d:', bool(np.isinf(X_p3d_full).any()))

# Duplicates (same feature vector across multiple FPs would hint at all-zero rows)
rows_ok = X_p3d_full[has_pocket == 1]
uniq = np.unique(rows_ok.round(4), axis=0)
print(f'unique pocket-3d vectors (among has_pocket=1): {len(uniq)}/{len(rows_ok)}')

# Acceptance gate #1
assert has_pocket.sum() >= 850, f'has_pocket={int(has_pocket.sum())} < 850 — extraction failed too often.'
assert float(atom_compl[has_pocket == 1].mean()) >= 0.90, f'atom_completeness mean too low'
print('\n✓ Integrity gate PASSED')

In [ ]:
# ── Cell 4: Per-block distribution histograms ──────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

blocks = {'A': [], 'B': [], 'C': [], 'D': []}
for j, n in enumerate(p3d_names):
    blocks[n[0]].append(j)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))
X_ok = X_p3d_full[has_pocket == 1]
for ax, (block, idx) in zip(axes, blocks.items()):
    vmat = X_ok[:, idx]
    rng = vmat.max(axis=0) - vmat.min(axis=0)
    vmat_n = (vmat - vmat.min(axis=0)) / (rng + 1e-8)
    ax.boxplot([vmat_n[:, k] for k in range(vmat_n.shape[1])], showfliers=False)
    ax.set_title(f'Block {block} ({len(idx)} features, min-max normalized)')
    ax.set_xticklabels([p3d_names[i].split('__')[1] for i in idx], rotation=90, fontsize=7)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(STRUCT_DIR / 'pocket3d_distributions.png'), dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 5: Correlation matrix — flag redundancy ──────────────────────────────────────────────────────────────────────
# Drop constant columns first (std=0 would give NaN correlation)
stds = X_ok.std(axis=0)
keep = stds > 1e-6
X_cor = X_ok[:, keep]
names_cor = [p3d_names[i] for i in np.where(keep)[0]]
C = np.corrcoef(X_cor, rowvar=False)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(C, vmin=-1, vmax=1, cmap='RdBu_r')
plt.colorbar(im, ax=ax)
ax.set_title(f'Pocket-3D feature correlation matrix ({C.shape[0]}x{C.shape[0]})')
plt.savefig(str(STRUCT_DIR / 'pocket3d_correlation.png'), dpi=120, bbox_inches='tight')
plt.show()

# High-correlation redundancy (|r| > 0.95, off-diagonal)
high = []
for i in range(len(names_cor)):
    for j in range(i + 1, len(names_cor)):
        if abs(C[i, j]) > 0.95:
            high.append((names_cor[i], names_cor[j], float(C[i, j])))
print(f'High-correlation pairs (|r|>0.95): {len(high)}')
for a, b, r in high[:20]:
    print(f'  {r:+.3f}  {a}  ↔  {b}')

In [ ]:
# ── Cell 6: Literature spot check on 5 canonical FPs ────────────────────────────────────────────────────────────────
# EGFP OH↔H148-ND1 expected 3.0–3.5 Å (anionic chromophore H-bond)
# Sirius OH in hydrophobic environment (no His nearby, correlates with 355-nm ex_max)
# DsRed phenol↔M163 <5 Å (characteristic red-FP packing)
# mTagBFP Y66-ring absent (chromophore precursor) or modified — atom_completeness should flag
# mCherry OH↔K163 NZ ~3.5 Å (proton acceptor in the DsRed family)

import build_pocket3d_features as bp3
import importlib; importlib.reload(bp3)
bp3.STRUCT_DIR = STRUCT_DIR_PDB

CANDIDATES = ['egfp', 'mcherry', 'dsred', 'sirius', 'mtagbfp']
print(f'{"slug":10s}  {"chrom":4s}  {"n_chrom":7s}  {"has_OH":6s}  {"OH-His":>6s}  {"OH-Asp/Glu":>10s}  {"ex_max":>7s}')
for slug in CANDIDATES:
    pdb = STRUCT_DIR_PDB / f'{slug}_minimized.pdb'
    if not pdb.exists():
        print(f'  {slug:10s}  -- PDB missing'); continue
    s = bp3.parse_pdb(pdb)
    if s is None:
        print(f'  {slug:10s}  -- parse failed'); continue
    oh = bp3._get(s, 'OH')
    has_oh = oh is not None
    if has_oh:
        d_his = bp3._nearest_named_atom_distance(s, oh, {'HIS'}, ('ND1','NE2'))
        d_dge = bp3._nearest_named_atom_distance(s, oh, {'ASP','GLU'}, ('OD1','OD2','OE1','OE2'))
    else:
        d_his = d_dge = float('nan')
    ex = meta.loc[meta['slug']==slug, 'ex_max']
    ex_val = float(ex.iloc[0]) if len(ex) else float('nan')
    print(f'  {slug:10s}  {str(s.chrom_resname):4s}  {len(s.chrom_atoms):7d}  {str(has_oh):6s}  '
          f'{d_his:6.2f}  {d_dge:10.2f}  {ex_val:7.1f}')

print('\nExpected: EGFP OH-His ~3.0-3.5 Å; Sirius OH-His >6 Å (no nearby His); mCherry OH-Asp/Glu ~3 Å.')

In [ ]:
# ── Cell 7: Correlation sanity — τ vs QY, electrostatic_proxy_OH vs ex_max ──────────────────────────────────────────────────────────
from scipy.stats import pearsonr

def idx_of(name):
    matches = [i for i, n in enumerate(p3d_names) if n.endswith(name)]
    return matches[0] if matches else None

mask = (has_p.flatten() == 1)

for feat_name, target, expected in [
    ('tau_cos',                   'qy',      'non-zero (planarity governs QY)'),
    ('tau_sin',                   'qy',      'non-zero'),
    ('oh_electrostatic_proxy_8A', 'ex_max',  'non-zero (electrostatic Stark effect)'),
    ('imid_electrostatic_proxy_8A','em_max', 'non-zero'),
    ('oh_nearest_his_nitrogen',   'ex_max',  'typically positive (anionic chromophore needs proton partner)'),
]:
    idx = idx_of(feat_name)
    if idx is None:
        print(f'  feature {feat_name} not found'); continue
    y = meta[target].values
    m = mask & ~np.isnan(y)
    if m.sum() < 50:
        print(f'  {feat_name} vs {target}: too few samples ({m.sum()})'); continue
    r, p = pearsonr(X_p3d[m, idx], y[m])
    print(f'  {feat_name:30s} vs {target:10s}  r={r:+.3f}  p={p:.2e}  (n={m.sum()})  | expected: {expected}')

# Any |r| > 0.10 on at least one pair is evidence the features carry signal.
# All near-zero ⇒ pocket3d is likely noise → consider dropping blocks.

In [ ]:
# ── Cell 8: Blue-cliff hypothesis — does pocket3d carry BFP-specific signal? ──────────────────────────────────────────────────────────────
# Blue FPs have em_max < 470 (Tier 2b n=41 cohort). Pocket3d Block A+B should
# distinguish protonated-phenol chemistry (BFP-like) from anionic (GFP-like).
# Quick test: can a single XGB tree on pocket3d separate blue vs green?

em = meta['em_max'].values
is_blue = (em > 0) & (em < 470)
is_green = (em >= 490) & (em <= 530)
print(f'blue FPs : {int(is_blue.sum())}  |  green FPs : {int(is_green.sum())}')

# Mean pocket3d feature differences (blue - green)
mean_diff = X_p3d[is_blue].mean(axis=0) - X_p3d[is_green].mean(axis=0)
top_diff = np.argsort(-np.abs(mean_diff))[:15]
print('\nTop pocket3d features with largest blue-vs-green mean shift:')
for i in top_diff:
    print(f'  {p3d_names[i]:50s}  Δ(blue-green)={mean_diff[i]:+.4f}')

In [ ]:
# ── Cell 9: Mini-XGBoost prep — build L / B / B+P matrices, prepare 5-fold splits ────────────────────────────────────────────────────────────
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

X_L = np.hstack([X_p3d, has_p])
X_B = X_lora
X_BP = np.hstack([X_lora, X_p3d, has_p])

print(f'L_pocket3d_only     : {X_L.shape}')
print(f'B_lora              : {X_B.shape}')
print(f'B_lora+pocket3d     : {X_BP.shape}')

# Default params (sensible for our sample size); we are probing direction, not ultimate MAE.
BASE_PARAMS = dict(n_estimators=600, max_depth=6, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.5,
                   reg_alpha=0.1, reg_lambda=1.0, min_child_weight=1,
                   tree_method='hist', device='cuda', random_state=42, n_jobs=1)

def mini_cv(X, y, n_folds=5):
    valid = ~np.isnan(y)
    Xv, yv = X[valid], y[valid]
    splits = list(KFold(n_splits=n_folds, shuffle=True, random_state=42).split(np.arange(len(yv))))
    fold_maes = []
    for tr, va in splits:
        m = XGBRegressor(**BASE_PARAMS)
        m.fit(Xv[tr], yv[tr], verbose=False)
        pred = np.clip(m.predict(Xv[va]), 300.0, 700.0)
        fold_maes.append(mean_absolute_error(yv[va], pred))
    return float(np.mean(fold_maes)), fold_maes

print('\nMini-CV harness ready.')

In [ ]:
# ── Cell 10: Mini-XGBoost gate on ex_max — decision point ────────────────────────────────────────────────────────────
import time
y_ex = meta['ex_max'].values.astype(np.float32)

t = time.time()
mae_L, fm_L   = mini_cv(X_L,  y_ex); print(f'  L_pocket3d_only  mean_fold MAE = {mae_L:6.3f}  ({time.time()-t:.0f}s)'); t = time.time()
mae_B, fm_B   = mini_cv(X_B,  y_ex); print(f'  B_lora           mean_fold MAE = {mae_B:6.3f}  ({time.time()-t:.0f}s)'); t = time.time()
mae_BP, fm_BP = mini_cv(X_BP, y_ex); print(f'  B_lora+pocket3d  mean_fold MAE = {mae_BP:6.3f}  ({time.time()-t:.0f}s)')

delta = mae_BP - mae_B
print(f'\n  Δ(B+P - B) = {delta:+.3f} nm  (negative = pocket3d helps)')

# Decision gate per plan
print('\n' + '='*72)
if delta <= -0.3:
    verdict = 'STRONG POSITIVE: pocket3d reduces MAE by >= 0.3 nm under random KFold.'
    action  = 'Proceed to full clustered benchmark. Expected: clustered-50 win likely.'
elif delta < 0.0:
    verdict = 'WEAK POSITIVE: pocket3d slightly helps under random KFold.'
    action  = 'Proceed to full clustered benchmark. Outcome uncertain — prepare fallback.'
elif delta < 0.3:
    verdict = 'NEUTRAL: pocket3d no-op within noise under random KFold.'
    action  = 'Proceed to clustered benchmark (clustered CV is a different regime — may still help).'
else:
    verdict = 'NEGATIVE: pocket3d HURTS under random KFold by >= 0.3 nm.'
    action  = 'ABORT full benchmark. Enter fallback ladder (drop D → keep only A+B → bump reg).'
print(f'VERDICT: {verdict}')
print(f'ACTION : {action}')
print('='*72)

In [ ]:
# ── Cell 11: Blue-cohort stratified MAE (secondary win condition) ──────────────────────────────────────────────────────────────
# Even if overall delta is ~0, a blue-cohort improvement is a separate publishable signal.
# Diagnostic Tier 2b: blue cohort (em_max<470, n=41) had +11-14 nm bias on B_lora.
def blue_stratified_mae(X, y):
    valid = ~np.isnan(y)
    em = meta['em_max'].values
    is_blue_v = valid & (em > 0) & (em < 470)
    Xv, yv = X[valid], y[valid]
    # Train on all, predict on all via 5-fold OOF
    from sklearn.model_selection import KFold
    pred = np.full(len(yv), np.nan)
    for tr, va in KFold(n_splits=5, shuffle=True, random_state=42).split(np.arange(len(yv))):
        m = XGBRegressor(**BASE_PARAMS)
        m.fit(Xv[tr], yv[tr], verbose=False)
        pred[va] = np.clip(m.predict(Xv[va]), 300, 700)
    blue_mask_in_valid = is_blue_v[valid]
    if blue_mask_in_valid.sum() < 10:
        return float('nan'), 0
    mae_blue = float(np.mean(np.abs(pred[blue_mask_in_valid] - yv[blue_mask_in_valid])))
    return mae_blue, int(blue_mask_in_valid.sum())

print('Blue-cohort (em_max<470) MAE on ex_max:')
mae_blue_B,  n_b = blue_stratified_mae(X_B,  y_ex); print(f'  B_lora          : {mae_blue_B:6.3f}  n={n_b}')
mae_blue_BP, _   = blue_stratified_mae(X_BP, y_ex); print(f'  B_lora+pocket3d : {mae_blue_BP:6.3f}')
delta_blue = mae_blue_BP - mae_blue_B
print(f'  Δ blue = {delta_blue:+.3f} nm   (negative = pocket3d helps blue)')
if delta_blue <= -2.0:
    print('  ✓ BLUE-CLIFF SIGNAL: pocket3d reduces blue-cohort MAE by >=2 nm (publishable even if overall is flat)')

In [ ]:
# ── Cell 12: Summary JSON for downstream use ─────────────────────────────────────────────────────────────────
import json
summary = {
    'has_pocket'           : int(has_pocket.sum()),
    'atom_completeness_mean': float(atom_compl[has_pocket == 1].mean()) if has_pocket.sum() else 0.0,
    'mini_mae_L'           : mae_L,
    'mini_mae_B'           : mae_B,
    'mini_mae_BP'          : mae_BP,
    'delta_overall_ex_max' : float(delta),
    'mini_mae_blue_B'      : mae_blue_B,
    'mini_mae_blue_BP'     : mae_blue_BP,
    'delta_blue_ex_max'    : float(delta_blue),
    'verdict'              : verdict,
    'action'               : action,
}
with open(STRUCT_DIR / 'pocket3d_validation.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))